# 01 — Pipeline smoke test on 5 known CLAGN

**Milestone 2 (critical early gate).** Before scaling to 10^5–10^6 objects, confirm that Fornax's light-curve collector recovers the *known* mid-IR (NEOWISE W1/W2) and optical (ZTF g) changing-look events for five well-documented CLAGN.

**Pass criterion:** for each target, the recovered light curve shows the expected event (see `expected_event` in `src/smoke_test_targets.py`) at roughly the published amplitude. If the **mid-IR-selected** slot-5 object is missed but the bright archetypes pass, that flags a sensitivity limit in our discovery channel — informative either way.

> Run this **on the Fornax Science Console** (default `python3` kernel), where archive access lives.

**API note:** the collector returns **flux in mJy** (not magnitudes), in a `MultiIndexDFObject` indexed by `[objectid, label, band, time]` with columns `flux`, `err`. Bands are `WISE_W1`, `WISE_W2`, `ztf_g/r/i`. We work in flux and convert to an instrumental magnitude (`-2.5*log10(flux)`) only for amplitude/trend metrics (zero-point cancels).

## 0. One-time setup on Fornax

Clone the helper modules and install the collector's requirements (run once per server; the `!` lines are shell commands):

```bash
cd ~ && git clone https://github.com/nasa-fornax/fornax-demo-notebooks.git
pip install -r ~/fornax-demo-notebooks/light_curves/requirements_light_curve_collector.txt
```

Then point `FORNAX_CODE_SRC` below at the cloned `code_src` directory.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))  # project src/

# fornax-demo-notebooks light-curve helper modules:
FORNAX_CODE_SRC = os.path.expanduser('~/fornax-demo-notebooks/light_curves/code_src')
assert os.path.isdir(FORNAX_CODE_SRC), f'clone fornax-demo-notebooks first (missing {FORNAX_CODE_SRC})'
sys.path.insert(0, FORNAX_CODE_SRC)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord

# fornax collector API
from data_structures import MultiIndexDFObject
from sample_selection import clean_sample
from wise_functions import wise_get_lightcurves
from ztf_functions import ztf_get_lightcurves

# project code
from src.smoke_test_targets import TARGETS
from src import selection

for t in TARGETS:
    print(f'{t.name:28s} {t.direction:9s} {t.expected_event}')

## 1. Build the sample table

`clean_sample` wants a list of `SkyCoord` and a parallel list of labels. Resolve each target by name (SIMBAD/NED) rather than trusting the cached fallback coords. Skip the slot-5 TODO until it's pinned.

We use `consolidate_nearby_objects=False` — our 5 targets are far apart and we want a 1:1 mapping preserved by `label`.

In [ ]:
coords_list, labels_list = [], []
for t in TARGETS:
    if t.name.startswith('TODO'):
        print(f'skipping unpinned slot: {t.name}')
        continue
    try:
        c = SkyCoord.from_name(t.name)
    except Exception as e:
        print(f'name resolution failed for {t.name} ({e}); using fallback coords')
        c = SkyCoord(ra=t.ra_deg, dec=t.dec_deg, unit='deg')
    coords_list.append(c)
    labels_list.append(t.name)

sample_table = clean_sample(coords_list, labels_list, consolidate_nearby_objects=False)
sample_table

## 2. Collect multi-wavelength light curves

This is the step you asked about. `radius` is the cone-search match radius **in arcseconds** (1.0 default; bump to ~2 for bright/extended hosts and re-check for blends). Each call returns a `MultiIndexDFObject`; `.append()` accumulates them. `.data` is the underlying multi-index DataFrame — `reset_index()` flattens it to columns `[objectid, label, band, time, flux, err]`.

In [ ]:
df_lc = MultiIndexDFObject()

# mid-IR: NEOWISE W1 + W2
df_lc.append(wise_get_lightcurves(sample_table, radius=2.0, bandlist=['WISE_W1', 'WISE_W2']))

# optical: ZTF (g/r/i)
df_lc.append(ztf_get_lightcurves(sample_table, radius=2.0))

lc = df_lc.data.reset_index()  # columns: objectid, label, band, time, flux, err
print('rows:', len(lc), '| bands:', sorted(lc['band'].unique()))
lc.head()

In [ ]:
# checkpoint so we don't re-query the archives on every rerun
os.makedirs('../data/interim', exist_ok=True)
lc.to_parquet('../data/interim/smoke_test_lightcurves.parquet')
# lc = pd.read_parquet('../data/interim/smoke_test_lightcurves.parquet')  # reload

## 3. Plot each target (W1, W2, ZTF g) and eyeball the event

Plotted in flux (mJy). A turn-on rises, a turn-off fades; mid-IR (WISE) lags the optical (ZTF) by months for real accretion changes.

In [ ]:
def plot_target(lc, label):
    sub = lc[lc['label'] == label]
    fig, ax = plt.subplots(figsize=(9, 4))
    for band, marker in [('WISE_W1', 'o'), ('WISE_W2', 's'), ('ztf_g', '.')]:
        b = sub[sub['band'] == band].sort_values('time')
        if len(b):
            ax.errorbar(b['time'], b['flux'], yerr=b['err'], fmt=marker,
                        ms=4, alpha=0.7, label=band)
    ax.set_xlabel('time (MJD)'); ax.set_ylabel('flux (mJy)'); ax.set_title(label)
    ax.legend()
    return fig

for label in labels_list:
    plot_target(lc, label)
plt.show()

## 4. Quantitative check vs. literature thresholds

Convert flux → instrumental magnitude (`-2.5*log10(flux)`; zero-point cancels for amplitudes/trends) and confirm each known CLAGN clears the prior thresholds in `src/selection.py`. Sanity check on the metrics, not the production selection (which is injection-recovery-calibrated).

In [ ]:
def _mag(flux):
    flux = np.asarray(flux, float)
    flux = np.where(flux > 0, flux, np.nan)
    return -2.5 * np.log10(flux)

def features_for(lc, label):
    sub = lc[lc['label'] == label]
    def band(b):
        return sub[sub['band'] == b].sort_values('time')
    w1, w2, g = band('WISE_W1'), band('WISE_W2'), band('ztf_g')
    return {
        'delta_w1': selection.delta_mag(_mag(w1['flux'])),
        'delta_w2': selection.delta_mag(_mag(w2['flux'])),
        'delta_g':  selection.delta_mag(_mag(g['flux'])),
        'wise_color_change': selection.wise_color_change(_mag(w1['flux']), _mag(w2['flux'])),
        'trend_w1': selection.monotonic_trend(w1['time'], _mag(w1['flux'])),
    }

results = {lab: features_for(lc, lab) for lab in labels_list}
res_df = pd.DataFrame(results).T
res_df['is_candidate'] = [selection.is_candidate(r) for r in results.values()]
res_df

## 5. Verdict

- [ ] All 4 archetypes recovered with expected events
- [ ] Mid-IR-selected slot-5 object recovered (once pinned)
- [ ] Metrics clear prior thresholds for all turn-on AND turn-off cases
- [ ] Runtime per object recorded → extrapolate compute-credit budget for the full sample

If yes → proceed to parent-sample assembly + `scale_up`. If the pipeline misses known events → redesign before scaling.